In [1]:
!pip install torch matplotlib

In [2]:
import torch
import matplotlib
print("Libraries installed successfully")

Libraries installed successfully


In [3]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [4]:
data = [
("Malware","malicious software that damages systems"),
("Phishing","attack that tricks users to steal passwords"),
("Ransomware","malware that encrypts files and demands money"),
("Exploit","code that uses system vulnerabilities"),
("Vulnerability","weakness in a computer system"),
("Firewall","security system that monitors network traffic"),
("Encryption","process of protecting data by encoding it"),
("Hashing","converting data into fixed digital fingerprint"),
("Zero Day","unknown vulnerability not patched yet"),
("SOC","security operations center monitoring threats")
]

In [5]:
text = ""

for term,desc in data:
    text += term + " : " + desc + "\n"

print(text)

Malware : malicious software that damages systems
Phishing : attack that tricks users to steal passwords
Ransomware : malware that encrypts files and demands money
Exploit : code that uses system vulnerabilities
Vulnerability : weakness in a computer system
Firewall : security system that monitors network traffic
Encryption : process of protecting data by encoding it
Hashing : converting data into fixed digital fingerprint
Zero Day : unknown vulnerability not patched yet
SOC : security operations center monitoring threats



In [6]:
chars = sorted(list(set(text)))

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

vocab_size = len(chars)

print(vocab_size)

38


In [7]:
encoded = torch.tensor(encode(text), dtype=torch.long)

x = encoded[:-1]
y = encoded[1:]

x = x.unsqueeze(0)
y = y.unsqueeze(0)

In [8]:
class CyberLLM(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,64)
        self.rnn = nn.GRU(64,128,batch_first=True)
        self.fc = nn.Linear(128,vocab_size)

    def forward(self,x):

        x = self.embedding(x)
        out,_ = self.rnn(x)
        out = self.fc(out)

        return out

In [9]:
model = CyberLLM(vocab_size)

optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

loss_fn = nn.CrossEntropyLoss()

In [ ]:
losses = []

for epoch in range(300):

    output = model(x)

    loss = loss_fn(output.view(-1,vocab_size), y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if epoch % 50 == 0:
        print("epoch:",epoch,"loss:",loss.item())

epoch: 0 loss: 3.6440281867980957


In [ ]:
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
print("Model trained successfully")

In [ ]:
cyber_text = """
Cybersecurity protects systems and networks from digital attacks.
Malware is malicious software designed to harm systems.
Phishing is a social engineering attack used to steal user data.
Ransomware encrypts files and demands payment to restore access.
A vulnerability is a weakness in a system.
Exploit code takes advantage of a vulnerability.
Zero-day vulnerabilities are unknown to developers.
A firewall protects networks from unauthorized access.
Encryption protects sensitive information from attackers.
Authentication verifies the identity of a user.
"""

with open("cyber_data.txt","w") as f:
    f.write(cyber_text)

In [ ]:
import torch
import torch.nn as nn

with open("cyber_data.txt","r") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

print("Vocabulary size:", vocab_size)
print("Dataset length:", len(data))

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self,vocab):
        super().__init__()
        self.embed = nn.Embedding(vocab,64)
        self.linear = nn.Linear(64,vocab)

    def forward(self,x):
        x = self.embed(x)
        x = self.linear(x)
        return x

model = TinyGPT(vocab_size)
print(model)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for step in range(200):

    idx = torch.randint(len(data)-1,(8,))

    x = torch.stack([data[i:i+1] for i in idx])
    y = torch.stack([data[i+1:i+2] for i in idx])

    logits = model(x)

    loss = loss_fn(logits.view(-1,vocab_size),y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print("step",step,"loss",loss.item())

In [ ]:
start = torch.tensor([[stoi['C']]])

for _ in range(200):

    logits = model(start)
    probs = torch.softmax(logits[:,-1],dim=-1)

    next_char = torch.multinomial(probs,1)

    start = torch.cat([start,next_char],dim=1)

print(decode(start[0].tolist()))